# Deep MNIST Classification With Learned Superposition Activations

This notebook runs on the **full MNIST** split from sklearn/OpenML (~70k samples). The first run downloads MNIST and caches it locally.

Training uses **JAX + Optax** when available (`pip install "qfun[gpu]"`, then install a [CUDA-enabled jaxlib](https://jax.readthedocs.io/en/latest/installation.html) on Linux or Windows for GPU). The same code path runs on CPU if only the CPU jaxlib is installed. Without JAX, set `use_jax=False` in the config (PennyLane autograd over the full set can be very slow).

Features are **standardized** and compressed with **PCA** before baselines and quantum models. Unlike notebook 10, this version uses a **deep** superposition-activation classifier with `hidden_layers = (6, 6)`, and every hidden nonlinearity is learned from a superposition profile.


In [ ]:
import sys
from pathlib import Path

for _p in (Path.cwd(), Path.cwd().parent):
    if (_p / "qfun").is_dir():
        _root = str(_p.resolve())
        if _root not in sys.path:
            sys.path.insert(0, _root)
        break

from qfun.datasets import load_classification_dataset, prepare_classification_split
from qfun.qfan._classification_benchmarks import (
    build_comparison_rows,
    display_baseline_suite,
    display_quantum_result,
    print_comparison_table,
    print_split_summary,
    plot_training_diagnostics,
    run_default_baseline_suite,
    run_quantum_experiment,
)


## Config

Defaults mirror notebook 10, but this notebook switches from a single hidden bank to `hidden_layers = (6, 6)`. `use_jax=True` enables minibatch Adam on JAX (CPU or GPU). Without JAX installed, set `use_jax=False` (expect long runtimes on ~52k training points).


In [ ]:
data_seed = 7
test_size = 0.25
pca_components = 32

hidden_layers = (6, 6)
n_qubits = 3
steps = 60
learning_rate = 0.04
log_every = 5
snapshot_interval = 5
eval_shots = 3_000

try:
    import jax  # noqa: F401

    use_jax = True
except ImportError:
    use_jax = False
    print('Tip: pip install "qfun[gpu]" for JAX training on full MNIST (much faster).')

batch_size = 1024


## 1. Load, standardize, and compress MNIST

`load_classification_dataset("mnist")` may download once via sklearn/OpenML. We stratify the train/test split on labels, **standardize** 784-dimensional pixels, and apply **PCA** to `pca_components` dimensions before any model training.


In [ ]:
mnist_dataset = load_classification_dataset("mnist")
mnist_split = prepare_classification_split(
    mnist_dataset,
    test_size=test_size,
    seed=data_seed,
    standardize=True,
    pca_components=pca_components,
)
class_names = mnist_split.target_names
print(f"Dataset size: {mnist_dataset.X.shape[0]} samples (full MNIST)")
print_split_summary(mnist_dataset.name, mnist_split)


## 2. Baselines


In [ ]:
baseline_results = run_default_baseline_suite(
    mnist_split,
    seed=data_seed,
    mlp_hidden_layer_sizes=(64,),
)
display_baseline_suite(baseline_results, class_names)


## 3. Deep Standard Superposition Activations


In [ ]:
standard_result = run_quantum_experiment(
    "standard",
    label="MNIST deep standard superposition activations",
    split=mnist_split,
    hidden_layers=hidden_layers,
    n_qubits=n_qubits,
    steps=steps,
    learning_rate=learning_rate,
    seed=data_seed,
    log_every=log_every,
    snapshot_interval=snapshot_interval,
    eval_shots=eval_shots,
    use_jax=use_jax,
    batch_size=batch_size,
)
display_quantum_result(standard_result, class_names)


### Deep Standard Training Process (Snapshots)


In [ ]:
plot_training_diagnostics(standard_result)


## 4. Deep Mode A Signed Superposition Activations


In [ ]:
mode_a_result = run_quantum_experiment(
    "mode_a",
    label="MNIST deep Mode A superposition activations",
    split=mnist_split,
    hidden_layers=hidden_layers,
    n_qubits=n_qubits,
    steps=steps,
    learning_rate=learning_rate,
    seed=data_seed,
    log_every=log_every,
    snapshot_interval=snapshot_interval,
    eval_shots=eval_shots,
    use_jax=use_jax,
    batch_size=batch_size,
)
display_quantum_result(mode_a_result, class_names)


### Deep Mode A Training Process (Snapshots)


In [ ]:
plot_training_diagnostics(mode_a_result)


## 5. Deep Mode B Signed Superposition Activations


In [ ]:
mode_b_result = run_quantum_experiment(
    "mode_b",
    label="MNIST deep Mode B superposition activations",
    split=mnist_split,
    hidden_layers=hidden_layers,
    n_qubits=n_qubits,
    steps=steps,
    learning_rate=learning_rate,
    seed=data_seed,
    log_every=log_every,
    snapshot_interval=snapshot_interval,
    eval_shots=eval_shots,
    use_jax=use_jax,
    batch_size=batch_size,
)
display_quantum_result(mode_b_result, class_names)


### Deep Mode B Training Process (Snapshots)


In [ ]:
plot_training_diagnostics(mode_b_result)


## 6. Final Comparison


In [ ]:
comparison_rows = build_comparison_rows(
    baseline_results,
    [standard_result, mode_a_result, mode_b_result],
)
print_comparison_table(comparison_rows)
